In [1]:
# 05_valuation.ipynb
# Valuation analysis: P/E, FCF Yield, EV/EBITDA
# Tesla vs BYD vs Ford (2022-2024)
# Requires: 01_data_collection.ipynb to have been run first

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle
import yfinance as yf
from datetime import datetime

# Load data
with open('../data/metrics.pkl', 'rb') as f:
    metrics = pickle.load(f)

with open('../data/raw_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Define companies and colors
tickers = {
    "Tesla": "TSLA",
    "BYD": "BYDDY",
    "Ford": "F"
}

colors = {
    "Tesla": "#E31937",
    "BYD": "#1DB954",
    "Ford": "#003DA5"
}

# Last complete fiscal year
LAST_COMPLETE_YEAR = 2024

print("Data loaded successfully")

Data loaded successfully


In [2]:
# Pull year-end stock prices for market cap calculation
# Using last trading day of each year as proxy for December 31

years = [2022, 2023, 2024]
prices = {}

for name, ticker in tickers.items():
    company = yf.Ticker(ticker)
    ticker_prices = {}
    
    for year in years:
        # Pull last 5 trading days of December to get year-end price
        start = f"{year}-12-26"
        end = f"{year}-12-31"
        hist = company.history(start=start, end=end)
        
        if not hist.empty:
            # Take the last available closing price
            ticker_prices[year] = hist["Close"].iloc[-1]
        else:
            ticker_prices[year] = None
            
    prices[name] = ticker_prices
    print(f"{name} year-end prices: {ticker_prices}")

Tesla year-end prices: {2022: np.float64(123.18000030517578), 2023: np.float64(248.47999572753906), 2024: np.float64(417.4100036621094)}
BYD year-end prices: {2022: np.float64(7.907253742218018), 2023: np.float64(8.939661979675293), 2024: np.float64(11.249223709106445)}
Ford year-end prices: {2022: np.float64(8.946955680847168), 2023: np.float64(10.3624267578125), 2024: np.float64(8.986583709716797)}


In [3]:
# Calculate market cap = share price × shares outstanding
market_cap = {}

for name in tickers.keys():
    balance = data[name]["balance_sheet"]
    
    # Get shares outstanding (Share Issued row)
    shares = balance.loc["Share Issued"]
    
    # Match year-end shares to year-end prices
    mc = {}
    for year in years:
        # Find the balance sheet entry for this year
        year_mask = shares.index.year == year
        if year_mask.any():
            shares_outstanding = shares[year_mask].values[0]
            price = prices[name][year]
            if price is not None:
                mc[year] = shares_outstanding * price
            else:
                mc[year] = None
    
    market_cap[name] = mc
    print(f"\n{name} Market Cap:")
    for year, mc_val in mc.items():
        if mc_val:
            print(f"  {year}: ${mc_val/1e9:.1f}B")


Tesla Market Cap:
  2022: $389.7B
  2023: $791.4B
  2024: $1342.4B

BYD Market Cap:
  2022: $69.1B
  2023: $78.1B
  2024: $98.2B

Ford Market Cap:
  2022: $35.8B
  2023: $41.2B
  2024: $35.6B


In [4]:
# Calculate valuation metrics
# P/E = Market Cap / Net Income
# FCF Yield = Free Cash Flow / Market Cap
# EV/EBITDA = (Market Cap + Total Debt - Cash) / EBITDA

valuation = {}

for name in tickers.keys():
    income = data[name]["income"]
    balance = data[name]["balance_sheet"]
    
    rows = []
    
    for year in years:
        year_mask = lambda df: df.index.year == year
        
        # Market cap
        mc = market_cap[name][year]
        
        # Net income
        ni = income.loc["Net Income"][income.loc["Net Income"].index.year == year].values[0]
        
        # Free cash flow
        fcf = data[name]["cashflow"].loc["Free Cash Flow"][
            data[name]["cashflow"].loc["Free Cash Flow"].index.year == year
        ].values[0]
        
        # Total debt
        debt = balance.loc["Total Debt"][balance.loc["Total Debt"].index.year == year].values[0]
        
        # Cash
        cash = balance.loc["Cash And Cash Equivalents"][
            balance.loc["Cash And Cash Equivalents"].index.year == year
        ].values[0]
        
        # EBITDA
        ebitda = income.loc["EBITDA"][income.loc["EBITDA"].index.year == year].values[0]
        
        # Calculate metrics
        pe = mc / ni if ni > 0 else None
        fcf_yield = (fcf / mc) * 100 if mc else None
        ev = mc + debt - cash
        ev_ebitda = ev / ebitda if ebitda > 0 else None
        
        rows.append({
            "Year": year,
            "P/E Ratio": round(pe, 1) if pe else None,
            "FCF Yield (%)": round(fcf_yield, 2) if fcf_yield else None,
            "EV/EBITDA": round(ev_ebitda, 1) if ev_ebitda else None
        })
    
    valuation[name] = pd.DataFrame(rows).set_index("Year")
    print(f"\n=== {name} VALUATION ===")
    print(valuation[name])


=== Tesla VALUATION ===
      P/E Ratio  FCF Yield (%)  EV/EBITDA
Year                                     
2022       31.0           1.94       21.5
2023       52.8           0.55       53.0
2024      188.3           0.27       91.1

=== BYD VALUATION ===
      P/E Ratio  FCF Yield (%)  EV/EBITDA
Year                                     
2022        4.2          62.82        0.9
2023        2.6          61.01        0.2
2024        2.4          36.76        0.3

=== Ford VALUATION ===
      P/E Ratio  FCF Yield (%)  EV/EBITDA
Year                                     
2022        NaN          -0.04       31.8
2023        9.5          16.23       14.2
2024        6.1          18.92       12.2


In [5]:
# Visualize valuation metrics
fig_val = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "P/E Ratio",
        "FCF Yield (%)",
        "EV/EBITDA"
    )
)

metrics_to_plot = ["P/E Ratio", "FCF Yield (%)", "EV/EBITDA"]

for col, metric in enumerate(metrics_to_plot, start=1):
    for name in valuation:
        # Drop NaN values for plotting
        plot_data = valuation[name][metric].dropna()
        
        fig_val.add_trace(go.Scatter(
            x=plot_data.index,
            y=plot_data.values,
            name=name,
            mode="lines+markers",
            line=dict(color=colors[name], width=2),
            marker=dict(size=8),
            legendgroup=name,
            showlegend=(col == 1)
        ), row=1, col=col)

fig_val.update_layout(
    title_text="Valuation Metrics — Tesla vs BYD vs Ford (2022–2024)",
    template="plotly_white",
    height=450,
    width=1200,
    hovermode="x unified"
)

# Fix x-axis ticks
for col in [1, 2, 3]:
    fig_val.update_xaxes(
        tickmode="array",
        tickvals=[2022, 2023, 2024],
        ticktext=["2022", "2023", "2024"],
        row=1, col=col
    )

fig_val.show()

In [6]:
# Valuation heatmap — 2024 snapshot
# Normalize valuation metrics for comparison
# Note: For P/E and EV/EBITDA, LOWER is cheaper (better value)
# For FCF Yield, HIGHER is cheaper (better value)
# We invert P/E and EV/EBITDA so green always means "better value"

val_snapshot = pd.DataFrame({
    name: {
        "P/E Ratio": valuation[name].loc[2024, "P/E Ratio"],
        "FCF Yield (%)": valuation[name].loc[2024, "FCF Yield (%)"],
        "EV/EBITDA": valuation[name].loc[2024, "EV/EBITDA"]
    }
    for name in valuation
}).T

# Invert P/E and EV/EBITDA so higher = better value across all metrics
val_normalized = val_snapshot.copy()
val_normalized["P/E Ratio"] = 1 / val_normalized["P/E Ratio"]
val_normalized["EV/EBITDA"] = 1 / val_normalized["EV/EBITDA"]

# Normalize to 0-100
val_normalized = (val_normalized - val_normalized.min()) / (val_normalized.max() - val_normalized.min()) * 100

print("2024 Valuation Snapshot (normalized, higher = better value):")
print(val_normalized)

2024 Valuation Snapshot (normalized, higher = better value):
        P/E Ratio  FCF Yield (%)   EV/EBITDA
Tesla    0.000000       0.000000    0.000000
BYD    100.000000     100.000000  100.000000
Ford    38.561187      51.109893    2.136744


In [7]:
import plotly.express as px

fig_val_heatmap = px.imshow(
    val_normalized,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    title="Valuation Snapshot 2024 — Who Is Cheapest? (Normalized 0–100, Higher = Better Value)",
    labels=dict(color="Value Score"),
    aspect="auto"
)

fig_val_heatmap.update_layout(
    template="plotly_white",
    height=300,
    width=700
)

fig_val_heatmap.show()

## Key Finding

The market is paying 188x earnings for Tesla's future promise while pricing BYD at 2.4x earnings despite superior cash generation.The gap is explained by geopolitical risk premium and growth expectations, not current financial performance.

**Valuation summary (2024):**
- BYD: cheapest on every valuation metric — P/E 2.4x, FCF Yield 36.8%, EV/EBITDA 0.3x
- Ford: middle ground — P/E 6.1x, FCF Yield 18.9%, EV/EBITDA 12.2x  
- Tesla: most expensive on every metric — P/E 188x, FCF Yield 0.27%, EV/EBITDA 91x

**The core tension:** Tesla's premium valuation is justified only if non-automotive catalysts (FSD, Optimus, energy storage) materialize into revenue. The financial statements show no evidence of this yet. 
BYD's discount is justified only if geopolitical risk is as severe as the market implies. The financial statements suggest it may be overpriced as a risk.

*This analysis is based on 3 years of annual financial data and excludes qualitative factors, forward guidance, and market context. It is an analytical exercise, not investment advice.*

In [8]:
# Save valuation data
with open('../data/valuation.pkl', 'wb') as f:
    pickle.dump(valuation, f)
    
print("Valuation data saved to data/valuation.pkl")

Valuation data saved to data/valuation.pkl
